# 📚 Social Network Intelligence — RAG Only

**Approach:** Retrieval-Augmented Generation (RAG) without GNN structural learning.  
**Paper:** *Graph Machine Learning in the Era of Large Language Models* — ACM TIST (doi:10.1145/3732786)  
**Paradigm:** Section 5 — *Graphs Augmenting LLMs* → GraphRAG / G-Retriever style  

---

## Architecture (Paper-aligned)

```
            User Query
                │
                ▼
┌─────────────────────────────────────┐
│         DUAL RETRIEVER              │
│  ┌──────────────┐ ┌───────────────┐ │
│  │ Graph Ret.   │ │ Vector Ret.   │ │
│  │ (Cypher/     │ │ (FAISS +      │ │
│  │  networkx)   │ │  MiniLM-L6)   │ │
│  └──────────────┘ └───────────────┘ │
│            RRF Fusion               │
└─────────────────────────────────────┘
                │ Retrieved subgraph +
                ▼   passages
┌──────────────────────────────────────┐
│  Gemini 2.5 Flash (LLM)              │
│  LangChain PromptTemplate +          │
│  PydanticOutputParser                │
└──────────────────────────────────────┘
                │
                ▼  
            Structured Answer
```

**Paper reference:** The survey identifies three LLM-Graph paradigms:  
- *LLM as Enhancer* (uses LLM to enrich graph features)  
- **LLM as Predictor with Graph-as-Context ← this notebook** (RAG over graph structure)  
- *GNNs + LLMs Integration* (Notebook 3)

Specifically implements the **G-Retriever** pattern: subgraph retrieval → verbalization → LLM generation.

In [19]:
# ─── 0. Install dependencies ────────────────────────────────────────────────
# Run once in your environment
# !pip install langchain langchain-google-genai google-generativeai \
#              sentence-transformers faiss-cpu networkx python-dotenv \
#              pydantic numpy scikit-learn

In [20]:
!pip install sentence_transformers

In [21]:
# ─── 1. Imports & Environment ───────────────────────────────────────────────
import os
import json
import numpy as np
import networkx as nx
from typing import List, Dict, Any, Tuple, Optional
from dataclasses import dataclass, field
from dotenv import load_dotenv

from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from sentence_transformers import SentenceTransformer

load_dotenv()  # loads GOOGLE_API_KEY from .env

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
print(f"Using model: {GEMINI_MODEL}")
print(f"API key present: {'GOOGLE_API_KEY' in os.environ}")

Using model: gemini-2.5-flash
API key present: True


## Part 1 — Social Graph Construction
Build an in-memory social graph using NetworkX. In production this would be backed by Neo4j.

In [22]:
# ─── 2. Build Social Graph (NetworkX) ──────────────────────────────────────

def build_social_graph() -> nx.DiGraph:
    """
    Construct a directed social graph with user nodes, posts, and relationships.
    Mirrors the Neo4j schema from the main project:
      (:User)-[:FRIEND]->(:User)
      (:User)-[:POSTED]->(:Post)
      (:User)-[:LIKED]->(:Post)
    """
    G = nx.DiGraph()

    # Users
    users = [
        {"id": "u1", "name": "Alice",   "bio": "AI researcher, graph enthusiast",      "followers": 4200},
        {"id": "u2", "name": "Bob",     "bio": "ML engineer, loves PyTorch and GNNs",  "followers": 1800},
        {"id": "u3", "name": "Carol",   "bio": "Data scientist, NLP specialist",        "followers": 3100},
        {"id": "u4", "name": "Dave",    "bio": "Software engineer, open source dev",    "followers": 900},
        {"id": "u5", "name": "Eve",     "bio": "Researcher in knowledge graphs",        "followers": 5500},
        {"id": "u6", "name": "Frank",   "bio": "PhD student in graph machine learning",  "followers": 650},
        {"id": "u7", "name": "Grace",   "bio": "Tech lead, distributed systems",         "followers": 2200},
        {"id": "u8", "name": "Heidi",   "bio": "AI blogger and educator",               "followers": 7800},
    ]
    for u in users:
        G.add_node(u["id"], type="user", **u)

    # Posts
    posts = [
        {"id": "p1", "title": "GraphSAGE for social recommendation",
         "content": "GraphSAGE enables inductive learning on large social graphs by sampling neighborhoods.",
         "topic": "GNN", "likes": 412, "age_hours": 3},
        {"id": "p2", "title": "RAG vs fine-tuning for graph tasks",
         "content": "Comparing retrieval-augmented generation with fine-tuning on knowledge graph QA tasks.",
         "topic": "RAG", "likes": 891, "age_hours": 1},
        {"id": "p3", "title": "Neo4j vector index tutorial",
         "content": "How to build a hybrid Cypher + vector search pipeline using Neo4j 5.x.",
         "topic": "Database", "likes": 234, "age_hours": 8},
        {"id": "p4", "title": "LLMs on graphs: survey 2024",
         "content": "Comprehensive review of GNN+LLM methods: LLM as Enhancer, Predictor, and Integration.",
         "topic": "Survey", "likes": 1502, "age_hours": 2},
        {"id": "p5", "title": "GNN-RAG: Graph Neural Retrieval",
         "content": "GNN-RAG uses GNNs to retrieve reasoning paths from KGs, boosting LLM QA performance.",
         "topic": "GNN", "likes": 675, "age_hours": 5},
        {"id": "p6", "title": "Knowledge graph completion with transformers",
         "content": "BERT-style encoders for relation prediction on large-scale knowledge graphs.",
         "topic": "KG", "likes": 318, "age_hours": 12},
    ]
    for p in posts:
        G.add_node(p["id"], type="post", **p)

    # FRIEND edges
    friendships = [
        ("u1", "u2"), ("u1", "u3"), ("u2", "u4"), ("u3", "u5"),
        ("u4", "u6"), ("u5", "u7"), ("u6", "u8"), ("u7", "u1"),
        ("u2", "u5"), ("u3", "u7"), ("u1", "u8"), ("u4", "u8"),
    ]
    for src, dst in friendships:
        G.add_edge(src, dst, rel="FRIEND")

    # POSTED edges
    posted = [("u1","p1"),("u3","p2"),("u5","p3"),("u8","p4"),("u2","p5"),("u7","p6")]
    for u, p in posted:
        G.add_edge(u, p, rel="POSTED")

    # LIKED edges
    liked = [
        ("u2","p4"),("u3","p4"),("u4","p4"),("u5","p4"),("u6","p4"),
        ("u1","p2"),("u7","p2"),("u8","p2"),("u1","p5"),("u3","p5"),("u5","p5"),
        ("u2","p1"),("u6","p1"),("u4","p3"),("u7","p3"),
    ]
    for u, p in liked:
        G.add_edge(u, p, rel="LIKED")

    return G

G = build_social_graph()
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Users: {len([n for n,d in G.nodes(data=True) if d['type']=='user'])}")
print(f"Posts: {len([n for n,d in G.nodes(data=True) if d['type']=='post'])}")

Graph: 14 nodes, 33 edges
Users: 8
Posts: 6


## Part 2 — Graph Retriever
Implements pattern-based subgraph retrieval aligned with the paper's *Graph-as-Context* paradigm.

In [23]:
# ─── 3. Graph Retriever ─────────────────────────────────────────────────────

class GraphRetriever:
    """
    Retrieves subgraphs from the social graph for different query intents.
    Paper reference: G-Retriever (He et al. 2024) — subgraph retrieval for LLM context.
    """

    def __init__(self, graph: nx.DiGraph):
        self.G = graph

    def get_user_neighborhood(self, user_id: str, hops: int = 2) -> Dict[str, Any]:
        """Extract k-hop neighborhood of a user (friend network)."""
        if user_id not in self.G:
            return {"nodes": [], "edges": [], "error": f"User {user_id} not found"}

        # BFS up to `hops`
        visited = {user_id}
        frontier = {user_id}
        for _ in range(hops):
            next_frontier = set()
            for node in frontier:
                for nbr in self.G.successors(node):
                    if nbr not in visited:
                        visited.add(nbr)
                        next_frontier.add(nbr)
            frontier = next_frontier

        subgraph = self.G.subgraph(visited)
        return self._serialize_subgraph(subgraph)

    def get_friends_of_friends(self, user_id: str) -> List[Dict]:
        """Find 2-hop FRIEND connections not already direct friends."""
        if user_id not in self.G:
            return []
        direct_friends = {
            dst for src, dst, data in self.G.out_edges(user_id, data=True)
            if data.get("rel") == "FRIEND"
        }
        fof = {}
        for friend in direct_friends:
            for fof_node in self.G.successors(friend):
                if fof_node != user_id and fof_node not in direct_friends:
                    d = self.G.nodes[fof_node]
                    if d.get("type") == "user":
                        if fof_node not in fof:
                            fof[fof_node] = {"id": fof_node, "mutual_count": 0, **d}
                        fof[fof_node]["mutual_count"] += 1

        return sorted(fof.values(), key=lambda x: x["mutual_count"], reverse=True)

    def get_trending_posts(self, top_k: int = 5) -> List[Dict]:
        """Retrieve trending posts by engagement velocity (likes / age_hours)."""
        posts = [
            (nid, data) for nid, data in self.G.nodes(data=True)
            if data.get("type") == "post"
        ]
        scored = []
        for nid, data in posts:
            age = max(data.get("age_hours", 1), 0.1)
            score = data.get("likes", 0) / age
            scored.append({"id": nid, "score": round(score, 2), **data})
        return sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]

    def get_shortest_path(self, src: str, dst: str) -> Dict[str, Any]:
        """Find shortest path between two users for connection explanation."""
        try:
            path = nx.shortest_path(self.G, src, dst)
            path_details = []
            for i in range(len(path) - 1):
                edge_data = self.G.get_edge_data(path[i], path[i+1])
                src_name = self.G.nodes[path[i]].get("name", path[i])
                dst_name = self.G.nodes[path[i+1]].get("name", path[i+1])
                path_details.append({
                    "from": src_name, "to": dst_name,
                    "rel": edge_data.get("rel", "CONNECTED")
                })
            return {"path": [self.G.nodes[n].get("name", n) for n in path],
                    "hops": len(path) - 1, "edges": path_details}
        except nx.NetworkXNoPath:
            return {"path": [], "hops": -1, "edges": [], "error": "No path found"}

    def get_influence_context(self, user_id: str) -> Dict[str, Any]:
        """Aggregate influence signals for a user."""
        if user_id not in self.G:
            return {}
        ud = self.G.nodes[user_id]
        friends = [dst for _, dst, d in self.G.out_edges(user_id, data=True) if d.get("rel")=="FRIEND"]
        posts = [dst for _, dst, d in self.G.out_edges(user_id, data=True) if d.get("rel")=="POSTED"]
        liked_posts = [dst for _, dst, d in self.G.out_edges(user_id, data=True) if d.get("rel")=="LIKED"]
        total_likes = sum(self.G.nodes[p].get("likes", 0) for p in posts)
        return {
            "user": ud, "friend_count": len(friends), "post_count": len(posts),
            "total_likes": total_likes, "liked_count": len(liked_posts),
            "friends": [self.G.nodes[f] for f in friends],
            "posts": [self.G.nodes[p] for p in posts],
        }

    def _serialize_subgraph(self, sg: nx.DiGraph) -> Dict[str, Any]:
        """Convert a subgraph to a serializable dict (for LLM context)."""
        nodes = [{"id": n, **data} for n, data in sg.nodes(data=True)]
        edges = [{"from": u, "to": v, "rel": d.get("rel", "EDGE")}
                 for u, v, d in sg.edges(data=True)]
        return {"nodes": nodes, "edges": edges,
                "node_count": len(nodes), "edge_count": len(edges)}


graph_retriever = GraphRetriever(G)

# Quick test
fof = graph_retriever.get_friends_of_friends("u1")
print("Friends-of-friends for Alice:")
for f in fof:
    print(f"  {f['name']} (mutual connections: {f['mutual_count']})")

Friends-of-friends for Alice:
  Eve (mutual connections: 2)
  Dave (mutual connections: 1)
  Grace (mutual connections: 1)


## Part 3 — Vector Retriever
Dense retrieval over user profiles and post content using SentenceTransformers.

In [24]:
# ─── 4. Vector Retriever ────────────────────────────────────────────────────

class VectorRetriever:
    """
    Dense semantic retrieval using SentenceTransformer + cosine similarity.
    Paper reference: Section 5 — Embedding-based graph augmentation for LLMs.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"Loading SentenceTransformer: {model_name}")
        self.encoder = SentenceTransformer(model_name)
        self._user_texts: List[str] = []
        self._user_meta: List[Dict] = []
        self._post_texts: List[str] = []
        self._post_meta: List[Dict] = []
        self._user_embeddings: Optional[np.ndarray] = None
        self._post_embeddings: Optional[np.ndarray] = None

    def build_index(self, graph: nx.DiGraph):
        """Embed all users and posts from the graph."""
        for nid, data in graph.nodes(data=True):
            if data.get("type") == "user":
                text = f"User: {data.get('name','')}. Bio: {data.get('bio','')}"
                self._user_texts.append(text)
                self._user_meta.append({"id": nid, **data})
            elif data.get("type") == "post":
                text = f"{data.get('title','')}. {data.get('content','')} Topic: {data.get('topic','')}"
                self._post_texts.append(text)
                self._post_meta.append({"id": nid, **data})

        if self._user_texts:
            self._user_embeddings = self.encoder.encode(
                self._user_texts, normalize_embeddings=True, show_progress_bar=False
            )
        if self._post_texts:
            self._post_embeddings = self.encoder.encode(
                self._post_texts, normalize_embeddings=True, show_progress_bar=False
            )
        print(f"Indexed {len(self._user_texts)} users, {len(self._post_texts)} posts")

    def search_users(self, query: str, top_k: int = 5) -> List[Dict]:
        """Find semantically similar users."""
        if self._user_embeddings is None:
            return []
        q_emb = self.encoder.encode([query], normalize_embeddings=True)[0]
        scores = self._user_embeddings @ q_emb
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [{**self._user_meta[i], "similarity": float(scores[i])} for i in top_idx]

    def search_posts(self, query: str, top_k: int = 5) -> List[Dict]:
        """Find semantically similar posts."""
        if self._post_embeddings is None:
            return []
        q_emb = self.encoder.encode([query], normalize_embeddings=True)[0]
        scores = self._post_embeddings @ q_emb
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [{**self._post_meta[i], "similarity": float(scores[i])} for i in top_idx]


vector_retriever = VectorRetriever()
vector_retriever.build_index(G)

# Quick test
results = vector_retriever.search_posts("graph neural network recommendation")
print("\nTop posts for 'graph neural network recommendation':")
for r in results[:3]:
    print(f"  [{r['similarity']:.3f}] {r['title']}")

Loading SentenceTransformer: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 8 users, 6 posts

Top posts for 'graph neural network recommendation':
  [0.639] GraphSAGE for social recommendation
  [0.510] GNN-RAG: Graph Neural Retrieval
  [0.488] LLMs on graphs: survey 2024


## Part 4 — Hybrid Retrieval with RRF Fusion
Reciprocal Rank Fusion combines graph traversal + semantic search results.

In [25]:
# ─── 5. RRF Fusion ──────────────────────────────────────────────────────────

def reciprocal_rank_fusion(
    graph_results: List[Dict],
    vector_results: List[Dict],
    graph_weight: float = 0.6,
    vector_weight: float = 0.4,
    k: int = 60,
) -> List[Dict]:
    """
    Fuse graph and vector retrieval results using Reciprocal Rank Fusion.
    
    RRF score = Σ  w × (1 / (k + rank))
    
    Paper reference: Hybrid retrieval paradigm from GraphRAG survey
    (Pan et al. 2024, doi:10.1145/3777378)
    """
    scores: Dict[str, float] = {}
    entity_map: Dict[str, Dict] = {}

    for rank, item in enumerate(graph_results):
        eid = item.get("id", str(rank))
        scores[eid] = scores.get(eid, 0) + graph_weight * (1.0 / (k + rank + 1))
        entity_map[eid] = {**item, "source": "graph"}

    for rank, item in enumerate(vector_results):
        eid = item.get("id", str(rank))
        sim_boost = 1 + item.get("similarity", 0.5)
        scores[eid] = scores.get(eid, 0) + vector_weight * (1.0 / (k + rank + 1)) * sim_boost
        if eid in entity_map:
            entity_map[eid]["source"] = "hybrid"
            entity_map[eid]["similarity"] = item.get("similarity")
        else:
            entity_map[eid] = {**item, "source": "vector"}

    fused = sorted(
        [{**entity_map[eid], "rrf_score": round(scores[eid], 6)} for eid in scores],
        key=lambda x: x["rrf_score"], reverse=True
    )
    return fused


# Test fusion
g_results = graph_retriever.get_friends_of_friends("u1")
v_results = vector_retriever.search_users("machine learning researcher", top_k=5)
fused = reciprocal_rank_fusion(g_results, v_results)
print("Fused friend recommendations for Alice:")
for item in fused[:5]:
    print(f"  {item.get('name','?')} | RRF={item['rrf_score']:.5f} | source={item['source']}")

Fused friend recommendations for Alice:
  Eve | RRF=0.01850 | source=hybrid
  Frank | RRF=0.01018 | source=vector
  Dave | RRF=0.00968 | source=graph
  Grace | RRF=0.00952 | source=graph
  Carol | RRF=0.00939 | source=vector


## Part 5 — Pydantic Output Schemas
Structured output schemas per intent type — LangChain PydanticOutputParser pattern.

In [26]:
# ─── 6. Pydantic Output Schemas ─────────────────────────────────────────────

class FriendRecommendation(BaseModel):
    """RAG-based friend recommendation output."""
    recommended_users: List[str] = Field(description="Names of recommended users in ranked order")
    reasoning: str = Field(description="Why these users are recommended based on graph structure")
    connection_paths: List[str] = Field(description="How user connects to each recommendation via the graph")
    confidence: str = Field(description="Confidence level: high, medium, or low")

class InfluenceAnalysis(BaseModel):
    """RAG-based influence analysis output."""
    influence_score: float = Field(description="Estimated influence score from 0.0 to 1.0")
    network_role: str = Field(description="Role: regular_user, influencer, content_creator, or community_hub")
    key_factors: List[str] = Field(description="Top factors driving this user's influence")
    summary: str = Field(description="One paragraph summary of this user's network position")

class TrendingAnalysis(BaseModel):
    """RAG-based trending content analysis."""
    trending_posts: List[str] = Field(description="Titles of trending posts in ranked order")
    dominant_topics: List[str] = Field(description="Most active topic areas right now")
    engagement_insight: str = Field(description="Why these topics are trending now")
    peak_post: str = Field(description="The single highest-engagement post and why")

class ConnectionExplanation(BaseModel):
    """RAG-based connection explanation."""
    path_description: str = Field(description="Natural language description of the connection path")
    connection_strength: str = Field(description="strong, moderate, or weak")
    shared_interests: List[str] = Field(description="Topics or interests both users share")
    recommendation: str = Field(description="Should they connect and why")

print("Output schemas defined: FriendRecommendation, InfluenceAnalysis, TrendingAnalysis, ConnectionExplanation")

Output schemas defined: FriendRecommendation, InfluenceAnalysis, TrendingAnalysis, ConnectionExplanation


## Part 6 — RAG Pipeline with Gemini
Implements the full RAG chain: retrieve context → verbalize graph → generate with Gemini.

In [27]:
# ─── 7. RAG Pipeline ────────────────────────────────────────────────────────

class SocialGraphRAG:
    """
    Pure RAG pipeline for social network intelligence.
    NO GNN structural learning — retrieval only.
    
    Paper paradigm: LLM as Predictor with Graph-as-Context (Section 5).
    Pattern: G-Retriever (subgraph → verbalize → LLM generation).
    """

    def __init__(
        self,
        graph_ret: GraphRetriever,
        vector_ret: VectorRetriever,
        model_name: str = GEMINI_MODEL,
    ):
        self.graph_ret = graph_ret
        self.vector_ret = vector_ret
        self.llm = ChatGoogleGenerativeAI(model=model_name)
        self.str_parser = StrOutputParser()

    def _verbalize_graph_context(self, context: Any) -> str:
        """Convert graph retrieval results to natural language for the LLM prompt."""
        if isinstance(context, list):
            return json.dumps(context[:8], indent=2, default=str)
        elif isinstance(context, dict):
            return json.dumps(context, indent=2, default=str)
        return str(context)

    def _run_chain(self, schema_class, template_str: str, variables: dict) -> BaseModel:
        """Run: PromptTemplate | Gemini | PydanticOutputParser."""
        parser = PydanticOutputParser(pydantic_object=schema_class)
        prompt = PromptTemplate(
            template=template_str,
            input_variables=list(variables.keys()),
            partial_variables={"format_instructions": parser.get_format_instructions()},
        )
        chain = prompt | self.llm | parser
        try:
            return chain.invoke(variables)
        except Exception as e:
            # Fallback to str output
            print(f"  [Pydantic parse failed: {e}] — falling back to StrOutputParser")
            fallback = PromptTemplate(
                template=template_str.split("\n{format_instructions}")[0],
                input_variables=list(variables.keys()),
            )
            chain2 = fallback | self.llm | self.str_parser
            return chain2.invoke(variables)

    # ── Public API ───────────────────────────────────────────────────────────

    def recommend_friends(self, user_id: str, top_k: int = 5) -> FriendRecommendation:
        """
        RAG friend recommendation:
        1. Graph: 2-hop FRIEND traversal
        2. Vector: semantic user similarity
        3. RRF fusion → verbalize → Gemini
        """
        user_data = self.graph_ret.G.nodes.get(user_id, {})
        g_results = self.graph_ret.get_friends_of_friends(user_id)
        v_results = self.vector_ret.search_users(
            f"{user_data.get('bio', '')} {user_data.get('name', '')}", top_k=top_k
        )
        fused = reciprocal_rank_fusion(g_results, v_results)[:top_k]

        template = """\
You are a social network intelligence system using RAG (Retrieval-Augmented Generation).
You have NO learned GNN embeddings — you reason ONLY from the retrieved graph context below.

Target User: {user_name} (ID: {user_id})
User Bio: {user_bio}

Retrieved Friend Candidates (from 2-hop graph traversal + semantic similarity, RRF fused):
{candidates}

Task: Recommend the best friends for this user based ONLY on the retrieved context.

{format_instructions}"""

        return self._run_chain(FriendRecommendation, template, {
            "user_name": user_data.get("name", user_id),
            "user_id": user_id,
            "user_bio": user_data.get("bio", ""),
            "candidates": self._verbalize_graph_context(fused),
        })

    def analyze_influence(self, user_id: str) -> InfluenceAnalysis:
        """RAG influence analysis using graph-retrieved signals only."""
        context = self.graph_ret.get_influence_context(user_id)

        template = """\
You are analyzing user influence using ONLY retrieved graph signals (RAG approach — no GNN).

User Profile & Graph Context:
{context}

Analyze influence based purely on the structural signals in the context above.

{format_instructions}"""

        return self._run_chain(InfluenceAnalysis, template, {
            "context": self._verbalize_graph_context(context),
        })

    def get_trending(self, topic_hint: str = "") -> TrendingAnalysis:
        """RAG trending content analysis."""
        g_posts = self.graph_ret.get_trending_posts(top_k=6)
        v_posts = self.vector_ret.search_posts(
            topic_hint or "trending popular viral", top_k=5
        )
        fused = reciprocal_rank_fusion(g_posts, v_posts)

        template = """\
Analyze trending social media content using ONLY the retrieved post data below (RAG — no GNN).

Retrieved Posts (ranked by engagement velocity + semantic relevance, RRF fused):
{posts}

{format_instructions}"""

        return self._run_chain(TrendingAnalysis, template, {
            "posts": self._verbalize_graph_context(fused),
        })

    def explain_connection(self, user_a: str, user_b: str) -> ConnectionExplanation:
        """RAG connection explanation using shortest path + shared context."""
        path_info = self.graph_ret.get_shortest_path(user_a, user_b)
        ctx_a = self.graph_ret.get_influence_context(user_a)
        ctx_b = self.graph_ret.get_influence_context(user_b)

        # Find shared liked posts
        liked_a = {dst for _, dst, d in self.graph_ret.G.out_edges(user_a, data=True) if d.get("rel")=="LIKED"}
        liked_b = {dst for _, dst, d in self.graph_ret.G.out_edges(user_b, data=True) if d.get("rel")=="LIKED"}
        common_posts = [
            self.graph_ret.G.nodes[p].get("title", p)
            for p in liked_a & liked_b
        ]

        template = """\
Explain the connection between two social network users using ONLY retrieved graph data (RAG).

User A: {name_a} — {bio_a}
User B: {name_b} — {bio_b}

Shortest Path: {path}
Path Hops: {hops}
Common Liked Posts: {common_posts}

{format_instructions}"""

        return self._run_chain(ConnectionExplanation, template, {
            "name_a": ctx_a.get("user", {}).get("name", user_a),
            "bio_a": ctx_a.get("user", {}).get("bio", ""),
            "name_b": ctx_b.get("user", {}).get("name", user_b),
            "bio_b": ctx_b.get("user", {}).get("bio", ""),
            "path": " → ".join(path_info.get("path", [])),
            "hops": path_info.get("hops", "N/A"),
            "common_posts": ", ".join(common_posts) or "None found",
        })

rag_pipeline = SocialGraphRAG(graph_retriever, vector_retriever)
print("RAG pipeline ready")

RAG pipeline ready


## Part 7 — Run All Tasks

In [28]:
# ─── 8. Task 1: Friend Recommendation ──────────────────────────────────────
print("=" * 60)
print("TASK 1: Friend Recommendation (RAG Only)")
print("=" * 60)

recs = rag_pipeline.recommend_friends("u1", top_k=3)
print(f"\nRecommended for Alice:")
if hasattr(recs, "recommended_users"):
    for user in recs.recommended_users:
        print(f"  • {user}")
    print(f"\nReasoning: {recs.reasoning}")
    print(f"Confidence: {recs.confidence}")
else:
    print(recs)

TASK 1: Friend Recommendation (RAG Only)

Recommended for Alice:
  • Eve
  • Frank

Reasoning: Eve is the top recommendation due to her highest RRF score among the valid candidates, indicating a strong combined signal from both structural (2 mutual friends) and semantic similarity. Her bio as a 'Researcher in knowledge graphs' strongly aligns with Alice's interest as an 'AI researcher, graph enthusiast'. Frank is the second recommendation, also showing strong semantic alignment with Alice's interests, being a 'PhD student in graph machine learning', and was identified through a 2-hop traversal. While his semantic similarity score is higher than Eve's, his overall RRF score is lower, and no explicit mutual friends are mentioned.
Confidence: high


In [29]:
# ─── 9. Task 2: Influence Analysis ──────────────────────────────────────────
print("=" * 60)
print("TASK 2: Influence Analysis (RAG Only)")
print("=" * 60)

influence = rag_pipeline.analyze_influence("u8")  # Heidi — most followers
if hasattr(influence, "network_role"):
    print(f"\nUser: Heidi")
    print(f"Role: {influence.network_role}")
    print(f"Influence Score: {influence.influence_score}")
    print(f"Key Factors: {influence.key_factors}")
    print(f"Summary: {influence.summary}")
else:
    print(influence)

TASK 2: Influence Analysis (RAG Only)

User: Heidi
Role: influencer
Influence Score: 0.85
Key Factors: ['Large follower base (7800 followers)', 'Exceptional content engagement (1502 likes on a single post within 2 hours)', 'Specialized expertise as an AI blogger and educator']
Summary: Heidi is a highly influential user, primarily operating as a content creator and educator in the AI domain. Her substantial follower count of 7800, combined with the extraordinary engagement on her most recent post, which garnered 1502 likes in just 2 hours, demonstrates her ability to produce impactful and widely appreciated content. While her direct reciprocal network appears limited based on the provided context, her influence is clearly driven by her capacity to broadcast valuable information and command significant attention from her audience.


In [30]:
# ─── 10. Task 3: Trending Content ───────────────────────────────────────────
print("=" * 60)
print("TASK 3: Trending Posts (RAG Only)")
print("=" * 60)

trending = rag_pipeline.get_trending("AI graph learning")
if hasattr(trending, "trending_posts"):
    print(f"\nTop Trending:")
    for t in trending.trending_posts:
        print(f"  • {t}")
    print(f"\nDominant Topics: {trending.dominant_topics}")
    print(f"Engagement Insight: {trending.engagement_insight}")
    print(f"Peak Post: {trending.peak_post}")
else:
    print(trending)

TASK 3: Trending Posts (RAG Only)

Top Trending:
  • GraphSAGE for social recommendation
  • LLMs on graphs: survey 2024
  • RAG vs fine-tuning for graph tasks
  • GNN-RAG: Graph Neural Retrieval
  • Knowledge graph completion with transformers
  • Neo4j vector index tutorial

Dominant Topics: ['Graph Neural Networks (GNNs)', 'Large Language Models (LLMs)', 'Retrieval-Augmented Generation (RAG)', 'Knowledge Graphs (KGs)']
Engagement Insight: The trending content reveals a strong and active interest in the convergence of advanced AI technologies, particularly Graph Neural Networks (GNNs), Large Language Models (LLMs), and Retrieval-Augmented Generation (RAG). There is significant community engagement around how these paradigms can be integrated and applied to complex data structures like Knowledge Graphs (KGs) for tasks such as social recommendation, knowledge graph completion, and enhancing LLM performance in question-answering. The discussions also highlight practical considerations, 

In [31]:
# ─── 11. Task 4: Connection Explanation ─────────────────────────────────────
print("=" * 60)
print("TASK 4: Connection Explanation (RAG Only)")
print("=" * 60)

conn = rag_pipeline.explain_connection("u1", "u6")  # Alice → Frank
if hasattr(conn, "path_description"):
    print(f"\nAlice ↔ Frank:")
    print(f"Path: {conn.path_description}")
    print(f"Strength: {conn.connection_strength}")
    print(f"Shared Interests: {conn.shared_interests}")
    print(f"Recommendation: {conn.recommendation}")
else:
    print(conn)

TASK 4: Connection Explanation (RAG Only)

Alice ↔ Frank:
Path: Alice is connected to Frank through a 3-hop path: Alice knows Bob, Bob knows Dave, and Dave knows Frank.
Strength: moderate
Shared Interests: ['Artificial Intelligence', 'Graph Theory', 'Machine Learning']
Recommendation: Yes, they should connect. Their highly aligned professional interests in AI, graph theory, and machine learning indicate a strong potential for collaboration, knowledge sharing, or mentorship, despite the indirect social path and lack of common liked posts.


## Part 8 — Evaluation
Assess RAG retrieval quality without GNN predictions.

In [32]:
# ─── 12. RAG Evaluation Metrics ─────────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

print("=" * 60)
print("RAG EVALUATION")
print("=" * 60)

# Retrieval coverage: how many nodes does RAG retrieve vs total
total_users = len([n for n, d in G.nodes(data=True) if d["type"]=="user"])
total_posts  = len([n for n, d in G.nodes(data=True) if d["type"]=="post"])

retrieved_users = vector_retriever.search_users("researcher engineer", top_k=10)
retrieved_posts = vector_retriever.search_posts("machine learning", top_k=10)

user_recall  = len(retrieved_users) / total_users
post_recall  = len(retrieved_posts) / total_posts

print(f"\nRetrieval Coverage:")
print(f"  Users: {len(retrieved_users)}/{total_users} = {user_recall:.1%}")
print(f"  Posts: {len(retrieved_posts)}/{total_posts} = {post_recall:.1%}")

# RRF Score distribution
fused = reciprocal_rank_fusion(
    graph_retriever.get_friends_of_friends("u1"),
    vector_retriever.search_users("AI researcher", top_k=5)
)
print(f"\nRRF Fusion Score Distribution:")
print(f"  Top score:  {fused[0]['rrf_score']:.5f}" if fused else "  (empty)")
print(f"  Mean score: {np.mean([f['rrf_score'] for f in fused]):.5f}" if fused else "")

# Source distribution
sources = [f.get("source", "unknown") for f in fused]
from collections import Counter
print(f"  Sources: {dict(Counter(sources))}")

print("\n✓ RAG-only evaluation complete")
print("NOTE: This pipeline uses NO GNN — all intelligence comes from retrieval + Gemini generation.")

RAG EVALUATION

Retrieval Coverage:
  Users: 8/8 = 100.0%
  Posts: 6/6 = 100.0%

RRF Fusion Score Distribution:
  Top score:  0.01904
  Mean score: 0.01097
  Sources: {'hybrid': 1, 'vector': 4, 'graph': 2}

✓ RAG-only evaluation complete
NOTE: This pipeline uses NO GNN — all intelligence comes from retrieval + Gemini generation.
